<a href="https://colab.research.google.com/github/venkata18167/CSA6301---THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/36_Zero_Trust_Continuous_Verification_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 def zero_trust_authorize(request, resource_policy):
    """
    request:
    {
        "user": str,
        "mfa_passed": bool,
        "device_posture": {
            "antivirus_enabled": bool,
            "os_patched": bool
        },
        "resource": str
    }

    resource_policy:
    {
        resource: set(allowed_users)
    }

    Access is granted only if:
    1. User is authorized.
    2. MFA is completed.
    3. Antivirus is enabled.
    4. Operating System is fully patched.
    """

    reasons = []

    allowed_users = resource_policy.get(request["resource"], set())

    if request["user"] not in allowed_users:
        reasons.append("user not authorized for this resource")

    if not request["mfa_passed"]:
        reasons.append("MFA not completed")

    if not request["device_posture"]["antivirus_enabled"]:
        reasons.append("antivirus disabled")

    if not request["device_posture"]["os_patched"]:
        reasons.append("OS not fully patched")

    return {
        "granted": len(reasons) == 0,
        "reasons": reasons
    }

def test_experiment8():

    policy = {
        "finance_db": {
            "csmith",
            "afinance"
        }
    }
    healthy_request = {
        "user": "csmith",
        "mfa_passed": True,
        "device_posture": {
            "antivirus_enabled": True,
            "os_patched": True
        },
        "resource": "finance_db"
    }

    result1 = zero_trust_authorize(healthy_request, policy)

    print("Healthy Request:")
    print(result1)

    assert result1["granted"] is True

    unhealthy_request = dict(
        healthy_request,
        device_posture={
            "antivirus_enabled": False,
            "os_patched": True
        }
    )

    result2 = zero_trust_authorize(unhealthy_request, policy)

    print("\nAntivirus Disabled Request:")
    print(result2)

    assert result2["granted"] is False
    assert "antivirus disabled" in result2["reasons"]

    unauthorized_request = dict(
        healthy_request,
        user="attacker99"
    )

    result3 = zero_trust_authorize(
        unauthorized_request,
        policy
    )

    print("\nUnauthorized User Request:")
    print(result3)

    assert result3["granted"] is False

    print("\nAll test cases passed.")

test_experiment8()

Healthy Request:
{'granted': True, 'reasons': []}

Antivirus Disabled Request:
{'granted': False, 'reasons': ['antivirus disabled']}

Unauthorized User Request:
{'granted': False, 'reasons': ['user not authorized for this resource']}

All test cases passed.
